In [ ]:
import pandas as pd
from tqdm.notebook import tqdm as tq
import requests
user_agent = {'User-agent':'Mozilla/5.0 (Windows NT 10.0; Win64; x64; rv:120.0) Gecko/20100101 Firefox/120.0'}
import json
import random
import time
import datetime
pd.set_option('display.max_columns', None)

!pip install reg-normalizer
from reg_normalizer import RegionMatcher
matcher = RegionMatcher()

import os
os.chdir(r'/content/')

ModuleNotFoundError: No module named 'normalize'

# Доходы

In [ ]:
def get_new_income(file_old, file_new, date_from=None, date_to=None):
    # принимает название входного и выходного файлов (оба parquet) и ноль, одну или две даты в формате yyyy-mm

    old = pd.read_parquet(file_old)

    link = 'https://www.iminfin.ru/areas-of-analysis/budget/finansoviy-pasport-subjecta-rf/redirect/copen-imon/Data?uuid=77b109a3-1c64-42db-9f58-6d79b42ba198&dataVersion=07.03.2017 07.10.12.980&dsCode=TerritoryOnlySubject&TERRITORIES_paramPeriod=2014-05-28T00:00:00.000Z&_dc=1737443788154'
    r = requests.get(link, headers=user_agent)
    regs = r.json()['data']

    link = 'https://www.iminfin.ru/areas-of-analysis/budget/finansoviy-pasport-subjecta-rf/redirect/copen-imon/Data?uuid=77b109a3-1c64-42db-9f58-6d79b42ba198&dataVersion=07.03.2017 07.10.12.980&dsCode=ds_FK_Passport_MONTH_Periods&verified=true&latest=false&_dc=1737443788160'
    r = requests.get(link, headers=user_agent)
    dates = r.json()['data']
    first_date = dates[0][0]
    latest_date = dates[-1][0]

    dates_old = [f'{i[0]}-{i[1]:02d}' for i in old.drop_duplicates(['year', 'month'])[['year', 'month']].values]

    # если дата начала указана, проверяет, есть ли данные за этот период,
    # если не указана, принимаем за нее первый месяц, которого нет в файле
    if date_from:
        if date_from < first_date:
            print('Этих данных нет, первый доступный период —', first_date[:7])
            return
        elif date_from > latest_date:
            print('Этих данных еще нет, последний доступный период —', latest_date[:7])
            return
        else:
            date_from = [i for i in dates if i[0].startswith(date_from)][0]
    else:
        last_date_old = sorted(dates_old)[-1]
        date_from = dates[[(index, i) for index, i in enumerate(dates) if i[0].startswith(last_date_old)][0][0]+1]
    print(date_from)

    # если дата конца указана, проверяет, есть ли данные за этот период,
    # если не указана, принимаем за нее последний месяц, за который есть данные
    if date_to:
        if date_to < first_date:
            print('Этих данных нет, первый доступный период —', first_date[:7])
            return
        elif date_to > latest_date:
            print('Этих данных еще нет, последний доступный период —', latest_date[:7])
            return
        else:
            date_to = [i for i in dates if i[0].startswith(date_to)][0]
    else:
        date_to = dates[-1]
    print(date_to)

    dates_to_load = dates[dates.index(date_from):dates.index(date_to)+1]
    # print(dates_to_load)

    res = []
    for reg in regs[::30]:
        for date in dates_to_load:
            print(reg, date)
            link = f'https://www.iminfin.ru/areas-of-analysis/budget/finansoviy-pasport-subjecta-rf/redirect/copen-imon/Data?uuid=77b109a3-1c64-42db-9f58-6d79b42ba198&dataVersion=07.03.2017 07.10.12.980&dsCode=PassportFK_002_001_incomesDataAfter01052019&territory={reg[0]}&paramPeriod={date[0]}&_dc=1737443590492'
            r = requests.get(link, headers=user_agent)
            # print(r, link)
            temp = r.json()['data']
            print('temp', len(temp))
            for index, i in enumerate(temp):
                temp[index].extend([reg[0], reg[1], date[0][:4], date[0][5:7]])
            res.extend(temp)

    df = pd.DataFrame(res, columns=['income_part', 'plan', 'adj_plan_consolidated',
              'adj_plan_regional', 'adj_plan_growth_rate',
              'execution_consolidated',
              'execution_regional',
              'growth_rate_regional',
              'growth_rate_federal_district',
              'growth_rate_russia', 'income_level',
              'code', 'region', 'year', 'month'])
    df.loc[df['income_part']=='ИТОГО ДОХОДОВ ', 'income_level'] = 0
    df['year'] = df['year'].astype(int)
    df['month'] = df['month'].astype(int)

    df = matcher.match_dataframe(
        df,
        'region',
        weights={'levenshtein': 0.4, 'token_set': 0.6},
        approach_weights={'original': 0.3, 'stemmed': 0.7},
        threshold=70).drop(columns=['region', 'levenshtein_score'])
    df = matcher.attach_fields(df, 'ter', ['okato', 'oktmo', 'level'])
    df = df.rename(columns={'ter':'object_name', 'level':'object_level'})

    df = df[['year', 'month', 'income_level', 'income_part', 'plan', 'adj_plan_consolidated', 'adj_plan_regional',
           'adj_plan_growth_rate', 'execution_consolidated', 'execution_regional',
           'growth_rate_regional', 'growth_rate_federal_district',
           'growth_rate_russia', 'object_name', 'okato', 'oktmo', 'object_level']]

    df = pd.concat([old, df]).sort_values(['year', 'month']).drop_duplicates()
    df.to_parquet(file_new)

    return df

In [ ]:
get_new_income('test_parquet.parquet', 'test_parquet2.parquet', '2025-09')

# Расходы

In [ ]:
def get_new_expense(file_old, file_new, date_from=None, date_to=None):
    # принимает название входного и выходного файлов (оба parquet) и ноль, одну или две даты в формате yyyy-mm

    old = pd.read_parquet(file_old)

    link = 'https://www.iminfin.ru/areas-of-analysis/budget/finansoviy-pasport-subjecta-rf/redirect/copen-imon/Data?uuid=77b109a3-1c64-42db-9f58-6d79b42ba198&dataVersion=07.03.2017 07.10.12.980&dsCode=TerritoryOnlySubject&TERRITORIES_paramPeriod=2014-05-28T00:00:00.000Z&_dc=1737443788154'
    r = requests.get(link, headers=user_agent)
    regs = r.json()['data']

    link = 'https://www.iminfin.ru/areas-of-analysis/budget/finansoviy-pasport-subjecta-rf/redirect/copen-imon/Data?uuid=77b109a3-1c64-42db-9f58-6d79b42ba198&dataVersion=07.03.2017 07.10.12.980&dsCode=ds_FK_Passport_MONTH_Periods&verified=true&latest=false&_dc=1737443788160'
    r = requests.get(link, headers=user_agent)
    dates = r.json()['data']
    first_date = dates[0][0]
    latest_date = dates[-1][0]

    dates_old = [f'{i[0]}-{i[1]:02d}' for i in old.drop_duplicates(['year', 'month'])[['year', 'month']].values]

    # если дата начала указана, проверяет, есть ли данные за этот период,
    # если не указана, принимаем за нее первый месяц, которого нет в файле
    if date_from:
        if date_from < first_date:
            print('Этих данных нет, первый доступный период —', first_date[:7])
            return
        elif date_from > latest_date:
            print('Этих данных еще нет, последний доступный период —', latest_date[:7])
            return
        else:
            date_from = [i for i in dates if i[0].startswith(date_from)][0]
    else:
        last_date_old = sorted(dates_old)[-1]
        date_from = dates[[(index, i) for index, i in enumerate(dates) if i[0].startswith(last_date_old)][0][0]+1]
    print(date_from)

    # если дата конца указана, проверяет, есть ли данные за этот период,
    # если не указана, принимаем за нее последний месяц, за который есть данные
    if date_to:
        if date_to < first_date:
            print('Этих данных нет, первый доступный период —', first_date[:7])
            return
        elif date_to > latest_date:
            print('Этих данных еще нет, последний доступный период —', latest_date[:7])
            return
        else:
            date_to = [i for i in dates if i[0].startswith(date_to)][0]
    else:
        date_to = dates[-1]
    print(date_to)

    dates_to_load = dates[dates.index(date_from):dates.index(date_to)+1]
    print(dates_to_load)

    res = []
    for reg in regs[::30]:
        for date in dates_to_load:
            print(reg[1], date[0], len(res))
            link = f'https://www.iminfin.ru/areas-of-analysis/budget/finansoviy-pasport-subjecta-rf/redirect/copen-imon/Data?uuid=556b0bd7-00a9-4713-84cc-5b54bcd506c7&dataVersion=07.03.2017 07.11.21.305&dsCode=PassportFK_002_002_outcomesDataAfter01052019&territory={reg[0]}&PassportFK_002_002_outcomesType=2&paramPeriod={date[0]}&_dc=1765489263390'
            r = requests.get(link, headers=user_agent)
            # print(r, link)
            temp = r.json()['data']
            print('temp', len(temp))
            for index, i in enumerate(temp):
                temp[index].extend([reg[0], reg[1], date[0][:4], date[0][5:7]])
            res.extend(temp)

    df = pd.DataFrame(res, columns=['expense_part', 'plan', 'adj_plan_consolidated', 'adj_plan_regional',
               'adj_plan_growth_rate', 'execution_consolidated', 'execution_regional',
               'growth_rate_regional', 'growth_rate_federal_district',
               'growth_rate_russia', 'expense_level', 'okato', 'region', 'year', 'month']).drop(columns='okato')

    df.loc[df['expense_part']=='ИТОГО ДОХОДОВ ', 'expense_level'] = 0
    df['year'] = df['year'].astype(int)
    df['month'] = df['month'].astype(int)

    df = matcher.match_dataframe(
        df,
        'region',
        weights={'levenshtein': 0.4, 'token_set': 0.6},
        approach_weights={'original': 0.3, 'stemmed': 0.7},
        threshold=70).drop(columns=['region', 'levenshtein_score'])
    df = matcher.attach_fields(df, 'ter', ['okato', 'oktmo', 'level'])
    df = df.rename(columns={'ter':'object_name', 'level':'object_level'})

    df = df[['year', 'month', 'expense_level', 'expense_part', 'plan', 'adj_plan_consolidated', 'adj_plan_regional',
               'adj_plan_growth_rate', 'execution_consolidated', 'execution_regional',
               'growth_rate_regional', 'growth_rate_federal_district',
               'growth_rate_russia', 'object_name', 'okato', 'oktmo', 'object_level']]

    df = pd.concat([old, df]).sort_values(['year', 'month']).drop_duplicates()
    df.to_parquet(file_new)

    return df

In [ ]:
get_new_expense('test_expense.parquet', 'test_expense2.parquet', '2022-05', '2022-06')